# Tool Calling

In [2]:
import dotenv
from agents import Agent, ModelSettings, Runner, function_tool, trace

dotenv.load_dotenv()

True

Create a static calorie table that we can use as a tool:

In [7]:
@function_tool
def get_food_calories(food_item: str) -> str:
    """
    Get calorie information for common foods to help with nutrition tracking.

    Args:
        food_item: Name of the food (e.g., "apple", "banana")

    Returns:
        Calorie information per standard serving
    """
    # Simple calorie database - in real world, you'd use USDA API
    calorie_data = {
        "apple": "80 calories per medium apple (182g)",
        "banana": "105 calories per medium banana (118g)",
        "broccoli": "25 calories per 1 cup chopped (91g)",
        "almonds": "164 calories per 1oz (28g) or about 23 nuts",
    }

    food_key = food_item.lower()
    if food_key in calorie_data:
        return f"{food_item.title()}: {calorie_data[food_key]}"
    else:
        return f"I don't have calorie data for {food_item} in my database. Try common foods like apple, chicken breast, or rice."

In [11]:
@function_tool
def get_food_nutrition(food_item: str) -> str:
    """Get approximate nutrition information for a common food per serving."""
    nutrition_data = {
        "apple": "80 calories, 0.5g protein, 21g carbohydrates, 0.3g fat, 4g fiber per medium apple (182g)",
        "banana": "105 calories, 1.3g protein, 27g carbohydrates, 0.4g fat, 3.1g fiber per medium banana (118g)",
        "broccoli": "25 calories, 2.5g protein, 5g carbohydrates, 0.3g fat, 2.4g fiber per cup chopped (91g)",
        "almonds": "164 calories, 6g protein, 6g carbohydrates, 14g fat, 3.5g fiber per 1oz (28g)",
    }

    food_key = food_item.strip().lower()
    if food_key in nutrition_data:
        return f"{food_item.title()}: {nutrition_data[food_key]}"

    return f"I don't have nutrition data for {food_item}. Try apple, banana, broccoli, or almonds."

In [15]:
@function_tool
def get_leave_entitlement(leave_type: str) -> str:
    """Look up the annual entitlement for a leave type, or list all entitlements."""

    leave_data = {
        "paid": ("Paid leave", 10),
        "paid leave": ("Paid leave", 10),
        "sick": ("Sick leave", 5),
        "sick leave": ("Sick leave", 5),
        "casual": ("Casual leave", 3),
        "casual leave": ("Casual leave", 3),
    }

    leave_key = leave_type.strip().lower()

    if leave_key in {"all", "all leaves", "leave"}:
        return (
            "Annual leave entitlements: "
            "10 paid leaves, 5 sick leaves, and 3 casual leaves."
        )

    if leave_key in leave_data:
        name, days = leave_data[leave_key]
        return f"{name}: {days} days per year."

    return (
        f"I don't have information for '{leave_type}'. "
        "Try paid leave, sick leave, casual leave, or all."
    )

Let's test this out: 

_The following cell only works before you add the `@function_tool` annotation to `get_food_calories` function_

In [4]:
get_food_calories('banana')

'Banana: 105 calories per medium banana (118g)'

In [12]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    """,
    tools=[get_food_calories, get_food_nutrition],
    )

In [22]:
hr_agent = Agent(
    name="HR Assistant",
    instructions="""
    You are a helpful HR assistant.
    Use get_leave_entitlement to answer questions about annual leave entitlements.
    Be concise and distinguish annual entitlement from leave already taken.
    """,
    tools=[get_leave_entitlement],
)

In [23]:
with trace("HR Assistant with tools"):
    result = await Runner.run(
        hr_agent, "How many total leaves are there in the company?"
    )
    print(result.final_output)

Annual entitlement (paid leave): 10 days per year.

Leaves taken (this year): not available here. Want me to check current take-up for a specific employee?


In [13]:
with trace("Nutrition Assistant with tools"):
    result = await Runner.run(
        calorie_agent, "How many calories are in total in a banana and an apple also give me the protein content?"
    )
    print(result.final_output)

Total calories: 185 kcal (banana 105 + apple 80)

Total protein: 1.8 g (banana 1.3 g + apple 0.5 g)


In [10]:
with trace("Nutrition Assistant with tools"):
    result = await Runner.run(
        calorie_agent, "How many grams of banana will give me 105 calories?"
    )
    print(result.final_output)

Approximately 118 grams.

Calculation: 105 kcal × 100 g / 89 kcal ≈ 118 g (banana ≈ 89 kcal per 100 g). Actual calories vary with ripeness.


Enforce tools use:

In [ ]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    """,
    tools=[get_food_calories],
    model_settings=ModelSettings(tool_choice="get_food_calories"),
)

with trace("Nutrition Assistant with tools enforced"):
    result = await Runner.run(
        calorie_agent, "How many calories are in total in a banana and an apple?"
    )
    print(result.final_output)